# Hedonic hot & cold spots

3-D reconstruction of the **liking** circuitry described by
Berridge & Kringelbach (2015), *Pleasure systems in the brain*, **Neuron** 86, 646–664
(`references/liking_and_wanting/pleasure_systems_in_brain.pdf`), drawn on the Allen
Mouse Brain atlas.

**How to read the figure**

| element | meaning |
|---|---|
| 🔴 red | **hotspot** — opioid/orexin stimulation *amplifies* “liking” reactions |
| 🔵 blue | **coldspot** — the same stimulation *suppresses* “liking” |
| solid | causally **established** in rodents |
| translucent | **tentative** (brainstem) |
| translucent + wireframe | **potential** — suggested but not yet localized |
| faint grey | parent structure, for spatial context |
| faint purple | “wanting” generators (mesolimbic dopamine) — shown for contrast; the paper argues dopamine does **not** itself create “liking” |

Hotspots are tiny (~1 mm³ in rat, ≈10 % of the nucleus accumbens) and **do not coincide
with whole-structure atlas boundaries** — so each spot here is *carved* out of its parent
structure by anatomical fraction (rostral/caudal, dorsal/ventral, medial) to match the
verbal anatomy in the paper. See *Caveats* at the bottom.

This notebook produces three views: a **static labelled figure** (sagittal, paper-style),
and two **interactive** views you can rotate (inline k3d, and a full-quality pop-up).

## Method & resolution

The Allen atlas does not sub-parcellate these nuclei (no shell/core split, no rostro-caudal
divisions), so we build each spot directly from the annotation **voxels**:

1. take the voxel mask of the parent structure (e.g. `ACB`),
2. keep the subset matching the paper's description (e.g. *rostrodorsal quadrant of the
   medial shell*),
3. turn that submask into a smooth surface and place it in the scene.

**Resolution.** The spots are sub-millimetre, but the limiting factor is *not* atlas voxel
size — it is that the paper only gives *verbal* boundaries. `allen_mouse_50um` (50 µm voxels)
is plenty and is already installed; switch `ATLAS` to `allen_mouse_25um` for smoother
surfaces (downloads on first use).

In [ ]:
# ============================ configuration ============================
ATLAS        = "allen_mouse_50um"  # "allen_mouse_25um" -> smoother (downloads on 1st use)
CAMERA       = "frontal"           # interactive views: "frontal" | "sagittal" | "top" | "three_quarters"
SHOW_WANTING = True                # faint dopamine "wanting" generators, for contrast

import numpy as np
import vedo
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from types import SimpleNamespace
from brainglobe_atlasapi import BrainGlobeAtlas
from brainrender import Scene

atlas = BrainGlobeAtlas(ATLAS)
ANN   = atlas.annotation               # 3-D label volume, axes = (AP, SI, RL)
RES   = atlas.resolution[0]            # µm / voxel
MIDLINE_RL = ANN.shape[2] * RES / 2.0  # left-right midline, µm

HOT, COLD = "#e8413a", "#2b6cff"       # red = hot, blue = cold (as in the paper)
# confidence -> visual weight
STYLE = {
    "established": dict(alpha=0.95, wireframe=False),  # solid
    "tentative":   dict(alpha=0.55, wireframe=False),  # translucent
    "potential":   dict(alpha=0.28, wireframe=True),   # translucent + wireframe
}

In [ ]:
# ============================ helpers ============================
def region_mask(acronym):
    """Boolean voxel mask of a structure and all of its sub-structures."""
    ids = [atlas.structures[acronym]["id"]]
    ids += [atlas.structures[a]["id"] for a in atlas.get_structure_descendants(acronym)]
    return np.isin(ANN, ids)

def fractions(mask):
    """Per-voxel normalised position (0..1) along AP, SI, RL within `mask`,
    plus a `medial(frac)` selector (voxels within `frac` of the midline).
    Convention: 0 = rostral / dorsal / right."""
    ap, si, rl = np.where(mask)
    norm = lambda a: (a - a.min()) / max(a.max() - a.min(), 1)
    dist_mid = np.abs(rl * RES - MIDLINE_RL)
    return SimpleNamespace(
        ap=norm(ap.astype(float)), si=norm(si.astype(float)), rl=norm(rl.astype(float)),
        medial=lambda frac: dist_mid <= dist_mid.max() * frac,
    )

def submask(mask, keep):
    out = np.zeros_like(mask)
    out[np.where(mask)] = keep
    return out

# --- brainrender L-R quirk -------------------------------------------------
# brainrender exposes the scene in TWO frames that differ in the left-right sign:
#   * scene.renderables  (used for the inline k3d view) are mirrored to  +RL
#   * scene.plotter      (used for the pop-up and for screenshots) keeps  -RL
# So a custom mesh must be built with flip_z = +1 when shown via renderables,
# and flip_z = -1 when added to scene.plotter.  (Atlas voxels are +RL.)
def to_mesh(mask, flip_z, smooth=12):
    """Watertight surface of a voxel mask, in render space (AP, SI, flip_z*RL)."""
    xs, ys, zs = np.where(mask)
    p = 2
    sl = tuple(slice(c.min() - p, c.max() + p + 1) for c in (xs, ys, zs))
    origin = tuple(s.start * RES for s in sl)
    m = vedo.Volume(mask[sl].astype(float), spacing=(RES,) * 3, origin=origin).isosurface(0.5)
    if smooth:
        m = m.smooth(niter=smooth)
    v = m.vertices.copy(); v[:, 2] *= flip_z; m.vertices = v
    m.compute_normals()
    return m

def spot_mesh(spec, flip_z):
    """Build one hot/cold spot from its specification dict."""
    mask = region_mask(spec["parent"])
    if "select" in spec:
        mask = submask(mask, spec["select"](fractions(mask)))
    m = to_mesh(mask, flip_z)
    st = STYLE[spec["confidence"]]
    m.color(HOT if spec["kind"] == "hot" else COLD).alpha(st["alpha"]).lighting("plastic")
    if st["wireframe"]:
        m.wireframe(True)
    m.name = spec["name"]
    return m

def legend_handles():
    """Matplotlib legend patches matching the colour/confidence encoding."""
    return [
        Patch(facecolor=HOT,  alpha=0.95, label="hotspot - amplifies 'liking'  (established)"),
        Patch(facecolor=COLD, alpha=0.95, label="coldspot - suppresses 'liking' (established)"),
        Patch(facecolor=HOT,  alpha=0.55, label="hotspot - tentative (brainstem)"),
        Patch(facecolor=HOT,  alpha=0.28, hatch="///", label="hotspot - potential / not localized"),
        Patch(facecolor="#7e57c2", alpha=0.45, label="'wanting' generators (dopamine; not 'liking')"),
        Patch(facecolor="#bbbbbb", alpha=0.60, label="parent structure (context)"),
    ]

In [ ]:
# ===================== hedonic spots (from the paper) =====================
# Each spot is carved from its parent structure to match Berridge & Kringelbach (2015).
# select(F) returns a boolean over the parent's voxels (see `fractions` above):
#   F.ap / F.si / F.rl  : 0..1 position (0 = rostral / dorsal / right)
#   F.medial(frac)      : voxels within `frac` of the most-medial extent
SPOTS = [
    dict(name="NAc hotspot",  parent="ACB",  kind="hot",  confidence="established",
         note="rostrodorsal quadrant of medial shell (~1 mm3, ~10% of NAc)",
         select=lambda F: (F.ap < 0.42) & (F.si < 0.50) & F.medial(0.45)),
    dict(name="NAc coldspot", parent="ACB",  kind="cold", confidence="established",
         note="caudal half of medial shell",
         select=lambda F: (F.ap > 0.66) & F.medial(0.45)),
    dict(name="VP hotspot",   parent="PALv", kind="hot",  confidence="established",
         note="posterior VP; sufficient AND necessary for normal 'liking'",
         select=lambda F: F.ap > 0.70),
    dict(name="VP coldspot",  parent="PALv", kind="cold", confidence="established",
         note="rostral VP",
         select=lambda F: F.ap < 0.30),
    dict(name="Parabrachial hotspot", parent="PB", kind="hot", confidence="tentative",
         note="hindbrain (dorsal pons); 'appears able to contribute'"),
    dict(name="OFC hotspot",    parent="ORBm", kind="hot", confidence="potential",
         note="orbitofrontal; suggested, not localized in rodent"),
    dict(name="Insula hotspot", parent="AIv",  kind="hot", confidence="potential",
         note="agranular insula; suggested, not localized in rodent"),
]

# faint parent structures (spatial context) and "wanting" generators
CONTEXT = ["ACB", "PALv", "PB", "ORBm", "AIv"]
WANTING = ["VTA", "SNc"]   # dopamine source; NB dopamine \!= "liking"

In [ ]:
# ============================ build the scene ============================
def build_scene(flip_z):
    """flip_z = +1 for the inline k3d view (scene.renderables),
       flip_z = -1 for the pop-up / screenshots (scene.plotter).  See `to_mesh`."""
    scene = Scene(atlas_name=ATLAS, title="", inset=False)  # title/inset off -> clean figure
    for ac in CONTEXT:
        scene.add_brain_region(ac, alpha=0.05, color="#888888", silhouette=False)
    if SHOW_WANTING:
        for ac in WANTING:
            scene.add_brain_region(ac, alpha=0.15, color="#7e57c2", silhouette=False)
    spots = [spot_mesh(s, flip_z) for s in SPOTS]
    return scene, spots

In [ ]:
# ============== static labelled figure (sagittal, paper-style) ==============
# Renders a screenshot through brainrender (nice glass brain) then shows it inline
# with the legend underneath.  (A render window may flash briefly; it does not block.)
import os, tempfile

vedo.settings.default_backend = "vtk"
scene, spots = build_scene(flip_z=-1.0)            # scene.plotter frame
scene.render(camera="sagittal", interactive=False)  # "frontal" gives a coronal view
for s in spots:
    scene.plotter.add(s)
img_path = os.path.join(tempfile.gettempdir(), "hedonic_sagittal.png")
scene.plotter.screenshot(img_path)
scene.close()

img = plt.imread(img_path)
# autocrop the white margins
try:
    ink = (img[..., :3] < 0.97).any(axis=2)
    ys, xs = np.where(ink); m = 20
    img = img[max(ys.min()-m, 0):ys.max()+m, max(xs.min()-m, 0):xs.max()+m]
except Exception:
    pass

fig = plt.figure(figsize=(9, 8))
ax_img = fig.add_axes([0.0, 0.17, 1.0, 0.80]); ax_img.axis("off")
ax_img.imshow(img)
ax_img.set_title("Hedonic hot & cold spots - sagittal (after Berridge & Kringelbach 2015)",
                 fontsize=12)
ax_leg = fig.add_axes([0.0, 0.0, 1.0, 0.15]); ax_leg.axis("off")
ax_leg.legend(handles=legend_handles(), loc="center", ncol=2, frameon=False, fontsize=10)
plt.show()

In [ ]:
# ============================ spot key (text) ============================
print(f"{'spot':24s} {'parent':6s} {'kind':5s} {'confidence':12s} note")
print("-" * 96)
for s in SPOTS:
    print(f"{s['name']:24s} {s['parent']:6s} {s['kind']:5s} {s['confidence']:12s} {s.get('note', '')}")

In [ ]:
# ===================== inline view (k3d; drag to rotate) =====================
# Shown via scene.renderables -> build spots with flip_z = +1 (see `to_mesh`).
vedo.settings.default_backend = "k3d"
scene, spots = build_scene(flip_z=1.0)
scene.render(camera=CAMERA, interactive=False)

viewer = vedo.Plotter()                       # NB: not `plt` (that is matplotlib)
viewer.show(*scene.renderables, *spots)

In [ ]:
# ============== pop-up view (vtk, full quality; press 'q'/'Esc' to close) ==============
# Uses scene.plotter -> build spots with flip_z = -1 (see `to_mesh`).
vedo.settings.default_backend = "vtk"
scene, spots = build_scene(flip_z=-1.0)
scene.render(camera=CAMERA, interactive=False)
for s in spots:
    scene.plotter.add(s)

# in-window legend
from vedo import Sphere, LegendBox
_legend = [
    Sphere(r=1).c(HOT).alpha(0.95).legend("hotspot (established)"),
    Sphere(r=1).c(COLD).alpha(0.95).legend("coldspot (established)"),
    Sphere(r=1).c(HOT).alpha(0.55).legend("hotspot (tentative)"),
    Sphere(r=1).c(HOT).alpha(0.30).legend("hotspot (potential)"),
    Sphere(r=1).c("#7e57c2").legend("wanting (dopamine)"),
]
scene.plotter.add(LegendBox(_legend, width=0.22, height=0.30))
scene.plotter.show(interactive=True)

## Caveats & approximations

- **The spots are reconstructions, not atlas labels.** The paper gives verbal boundaries
  (*“rostrodorsal quadrant of the medial shell”*, *“posterior VP”*); these become fractional
  cuts of the parent structure. Tune the `select=` rules in `SPOTS` to taste.
- The Allen `ACB` is **not** split into shell/core, so the *medial shell* is approximated by
  the medial part of the accumbens.
- Most causal data are from **rat**; sizes are extrapolated onto the mouse atlas.
- `OFC`/`insula` hotspots are drawn as whole sub-regions because the paper does not localize
  them in rodent — hence the *potential* (wireframe) styling.
- brainrender returns `scene.renderables` mirrored (+RL) but `scene.plotter` un-mirrored
  (−RL); that is why the views build spots with opposite `flip_z` (handled for you).

## Towards the deeper version

Next steps for the “liking → wanting → action” end-product
(cf. `github.com/carlhenrikrolf/genaipedia.wiki` brain-areas article):

- drive the region list + colour/confidence from that **brain-areas table** instead of a
  hand-written `SPOTS` list (a column encoding the liking→wanting→action axis);
- add **connectivity** between highlighted areas — e.g. NAc⇄VP functional coupling and the
  VTA/SNc dopamine projections — as tubes/streamlines (`brainrender` can add cylinders/lines
  between region centroids, or render Allen mouse-connectivity projection data).